# Swish — modeling notebook

The readable version of `src/predict.py`. Same functions, run stage by stage with
the reasoning in between, so you can see what each step buys.

The question is: **given everything through a team's last game, do they win their next one?**
Predicting the current game from its own box score is trivial and useless — the final
score is right there in the features. Forecasting forward from prior form is the actual problem.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd

from src import features, predict
from src.config import N_FEATURES, ROLLING_WINDOW

---
## 1. The raw data

One row per team per game — so two rows per game — holding that game's own box score:
basic and advanced stats, as team totals and per-player maxima, with the opponent's
line joined on as `_opp` columns.

In [ ]:
games = features.load_games()
print(games.shape)
games.head()

## 2. The target

Each row is labelled with whether that team won its **next** game — `won` shifted
back one game within each team, in date order.

A team's last game in the dataset has no next game. Those rows are marked `2` so they
travel through the pipeline intact and get dropped at evaluation time rather than
silently distorting the label distribution.

In [ ]:
df = features.add_target(games)
df = features.drop_null_columns(df)

print(df["target"].value_counts().sort_index())
print(f"\n{df.shape[1]} columns survive the null check")

## 3. Rolling form — where the accuracy comes from

A single game's box score barely predicts the next one. A team's *recent form* does.

Every stat is re-expressed as a trailing 10-game mean, computed **within each
team-season**. Grouping by `(Team, season)` rather than by team alone is the whole
point: a window must never span two seasons, or October form would be contaminated
by the previous April.

The first nine games of each team's season have an incomplete window and are dropped.

Note that the per-game stats were min-max scaled by `load_games` before this step,
not after it. A 10-game average varies far less than a single game, so the rolling
columns land in a genuinely narrower band; re-scaling them afterwards would stretch
that band back out and hide the fact that an average is the steadier quantity.

In [ ]:
df = features.scale_features(df, features.feature_columns(df))
rolled = features.add_rolling_form(df, window=ROLLING_WINDOW)
before = len(rolled)
rolled = rolled.dropna().reset_index(drop=True)
print(f"{before:,} rows -> {len(rolled):,} after dropping incomplete windows")

rolling_cols = [c for c in rolled.columns if c.endswith(f"_{ROLLING_WINDOW}")]
print(f"{len(rolling_cols)} rolling columns")

## 4. The opponent

Form on its own is half the picture. Each row is joined against the rolling form of
the team it is *about to play*, so the model sees both sides of the upcoming matchup
before it happens.

After the merge, `_x` columns are the team's own form and `_y` columns are the
opponent's.

In [ ]:
rolled = features.add_next_game_context(rolled)
rolled = rolled.dropna(subset=["home_next", "Team_opp_next", "date_next"]).copy()
full = features.merge_opponent_form(rolled, window=ROLLING_WINDOW)

candidates = features.model_columns(full)
print(f"{full.shape[0]:,} rows x {len(candidates)} candidate features")
full[["Team_x", "Team_opp_next_x", "date_next", "home_next", "target"]].head()

## 5. Feature selection

Forward sequential selection narrows ~400 candidates to 30, scored under
`TimeSeriesSplit`. Chronological folds matter here as much as in the backtest — with
random folds the selector would pick features using seasons it is later graded on.

`RidgeClassifier` drives the selection because it is fast and selects fine; the final
model is different (see the next section).

This is the slow step, so its result is cached in `artifacts/selected_features.json`.
Pass `refresh=True` to re-run it.

In [ ]:
predictors = predict.load_or_select_features(full, candidates)

print(f"{len(predictors)} selected:")
for p in predictors:
    print(" ", p)

## 6. The model

`RidgeClassifier` emits hard labels only. That is a real limitation: there is no way
to say the model is 80% sure about one game and coin-flipping another, and no way to
check whether its confidence is calibrated.

`LogisticRegression` is the same kind of L2-regularized linear model — which is what
handles the heavy collinearity between rolling features — but it emits probabilities.

Validation is **walk-forward by season**: train on every prior season, test on the
next unseen one, roll forward. No shuffling, no future data in training. This is the
correct protocol for time-ordered sports data, and the main reason the number below
is trustworthy.

In [ ]:
raw_preds = predict.backtest(full, predict.make_model(), predictors)
print(f"{len(raw_preds):,} held-out predictions")
predict.score(raw_preds)

## 7. The double-prediction bug

Every game appears twice — once from each team's point of view — and the model has
never been told those two rows are the same event. It predicts them independently.

So it contradicts itself: on a meaningful share of games it says *both* teams win, or
*both* teams lose.

In [ ]:
print(f"contradictory games: {predict.contradiction_rate(raw_preds):.1%}")

### Reconciliation

For a game between A and B, one row estimates `p_A = P(A wins)` and the other
estimates `p_B = P(B wins)`. Two estimates of the same event, from opposite sides.
Average them:

```
p_A_final = (p_A + (1 - p_B)) / 2
p_B_final = 1 - p_A_final
```

The two now sum to 1 by construction, so contradictions are not merely rare —
they are structurally impossible.

Rows group by a symmetric key (the two team names sorted, plus the game date). A row
occasionally loses its partner in the merge and dropna steps upstream; those are
skipped and **counted**, not dropped silently.

In [ ]:
reconciled, unpaired = predict.reconcile(raw_preds)
print(f"unpaired rows skipped: {unpaired:,}")
print(f"contradictory games: {predict.contradiction_rate(reconciled):.1%}")
predict.score(reconciled)

## 8. Confidence tiers — the most interesting result

A single accuracy figure averages games the model is nearly certain about with games
that genuinely are coin flips. Splitting on `|p - 0.5|` shows the model knows the
difference between the two.

In [ ]:
tiers = predict.confidence_tiers(reconciled)
tiers["accuracy"] = (100 * tiers["accuracy"]).round(1)
tiers

## 9. Baselines

The bar that matters is **not** 50%. It is "just pick the home team". A model that
cannot clear that has not learned anything worth having.

The raw-box-score baseline uses the same pipeline without any feature engineering:
predict the next game from *this* game's stats. It is the measure of what steps 3
and 4 are actually worth.

In [ ]:
print(f"Coin flip                 50.0%")
print(f"Home-court advantage only {predict.home_court_baseline(raw_preds):.1%}")

In [ ]:
# Slow: runs its own feature selection over the un-engineered columns.
raw_baseline = predict.raw_feature_baseline()
print(f"Raw box score features    {raw_baseline['accuracy']:.1%}")

## 10. XGBoost benchmark

Reaching for a gradient-boosted tree is the obvious move, so it should be an evidenced
decision rather than a default. Same 30 features, same walk-forward splits, then again
on the full candidate set.

In [ ]:
for label, cols in [(f"{N_FEATURES} selected", predictors), (f"all {len(candidates)}", candidates)]:
    s = predict.score(predict.backtest(full, predict.make_xgb_model(), cols))
    print(f"XGBoost, {label:<16} acc {s['accuracy']:.1%}  AUC {s['auc']:.3f}")

**XGBoost loses**, on both feature sets. That is the real result and it is not tuned away.

The explanation is the shape of the data: ~400 rolling features that are heavily
collinear (a 10-game mean of `fg` and a 10-game mean of `fga` move together almost
perfectly) on ~15K rows. L2 regularization shrinks correlated coefficients together
and degrades gracefully; an unconstrained tree ensemble keeps splitting on
near-duplicate features and overfits at that feature-to-sample ratio.

### What was tested and rejected

Rest days, back-to-back flags, and trailing-7-day schedule density were all built and
evaluated. Combined lift was **+0.05 percentage points** — noise. They are deliberately
not in the model. A negative result is still a result, and shipping a feature that does
nothing is worse than not shipping it.